In [1]:
# ----------------------- EDIT THESE FOUR LINES ------------------------------
SMILES  = "OC(=O)c1ccc(O)cc1"    # copy exactly from the menu table
NAME    = "p-hydrobenzoic acid"    # short name, used in output filenames
SEED    = 230970                     # YOUR student ID number, digits only
NCONFS  = 100                         # tier floor: rigid 50, 1-3 rotors 100, flexible 200
# ----------------------------------------------------------------------------

WINDOW_KJ  = 12.6    # survivors = distinct minima within this window of the lowest
RMS_THRESH = 0.3     # angstrom, all-atom RMSD below which two minima are one
MAX_ITERS  = 2000    # force-field minimizer iterations per conformer
MAX_INP    = 6       # write input files for at most this many survivors (warn beyond)

import csv, sys, math
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdMolTransforms, Descriptors, rdMolDescriptors
from rdkit.Geometry import Point3D

KCAL_TO_KJ = 4.184
R_KCAL = 1.987204e-3   # kcal mol-1 K-1

In [2]:
def mirror_copy(mol):
    """Copy of mol with every conformer reflected through the yz plane."""
    m = Chem.Mol(mol)
    for conf in m.GetConformers():
        for i in range(m.GetNumAtoms()):
            p = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, Point3D(-p.x, p.y, p.z))
    return m

def unique_conformers(mol, energies, rms=RMS_THRESH, merge_mirror=None):
    """Return conformer ids sorted by energy, one representative per RMSD cluster.
    energies: dict conf_id -> energy (only converged conformers).
    All-atom, symmetry-aware RMSD (rdMolAlign.GetBestRMS). Mirror images are
    merged for achiral molecules; a chiral molecule's reflection never matches."""
    if merge_mirror is None:
        merge_mirror = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True,
                                                     useLegacyImplementation=False)) == 0
    mir = mirror_copy(mol) if merge_mirror else None
    order = sorted(energies, key=energies.get)
    kept, cluster_of = [], {}
    for cid in order:
        rep = None
        for k in kept:
            if rdMolAlign.GetBestRMS(mol, mol, cid, k) <= rms:
                rep = k; break
            if merge_mirror and rdMolAlign.GetBestRMS(mir, mol, cid, k) <= rms:
                rep = k; break
        if rep is None:
            kept.append(cid); cluster_of[cid] = cid
        else:
            cluster_of[cid] = rep
    return kept, cluster_of, merge_mirror

def orca_coords(mol, conf_id):
    """RDKit conformer -> ORCA coordinate block: '* xyz charge mult' ... '*'."""
    charge = Chem.GetFormalCharge(mol)
    mult = Descriptors.NumRadicalElectrons(mol) + 1
    conf = mol.GetConformer(conf_id)
    lines = [f"* xyz {charge} {mult}"]
    for atom in mol.GetAtoms():
        p = conf.GetAtomPosition(atom.GetIdx())
        lines.append(f"{atom.GetSymbol():<2s} {p.x:12.6f} {p.y:12.6f} {p.z:12.6f}")
    lines.append("*")
    return "\n".join(lines) + "\n"

def orca_input(mol, conf_id, title):
    """Full input for the course protocol: B3LYP/6-31+G(d,p), CPCM water, optimize and verify."""
    mult = Descriptors.NumRadicalElectrons(mol) + 1
    ks = "! UKS B3LYP" if mult > 1 else "! B3LYP"     # open shell needs UKS, stated explicitly
    head = (f"# {title[:100]}\n"
            f"{ks} 6-31+G(d,p) CPCM(Water) Opt Freq RIJCOSX def2/J\n"
            f"%pal nprocs 4 end\n%maxcore 3000\n%scf MaxIter 300 end\n")
    return head + orca_coords(mol, conf_id)

def torsion_list(mol):
    """Dihedrals worth auditing: one per rotatable bond, plus every X-O-H."""
    tors = []
    rot = Chem.MolFromSmarts("[!D1&!$(*#*)]-&!@[!D1&!$(*#*)]")
    for b, c in mol.GetSubstructMatches(rot):
        a = next((n.GetIdx() for n in mol.GetAtomWithIdx(b).GetNeighbors()
                  if n.GetIdx() != c and n.GetAtomicNum() > 1), None)
        d = next((n.GetIdx() for n in mol.GetAtomWithIdx(c).GetNeighbors()
                  if n.GetIdx() != b and n.GetAtomicNum() > 1), None)
        if a is not None and d is not None:
            tors.append(((a, b, c, d), f"rotatable bond {b}-{c}"))
    oh = Chem.MolFromSmarts("[H][OX2]-[*]~[*]")
    seen = set()
    for h, o, x, y in mol.GetSubstructMatches(oh):
        if (h, o, x) in seen:
            continue                      # one reference neighbour per hydroxyl
        seen.add((h, o, x))
        tors.append(((h, o, x, y), f"hydroxyl H-O{o}-{x}-{y}"))
    return tors

In [3]:
print(f"RDKit {rdkit.__version__} | molecule: {NAME} | seed: {SEED} | requested: {NCONFS}")
mol = Chem.MolFromSmiles(SMILES)
assert mol is not None, "SMILES did not parse; re-copy it from the menu"
canon, ikey = Chem.MolToSmiles(mol), Chem.MolToInchiKey(mol)
formula = rdMolDescriptors.CalcMolFormula(mol)
charge, mult = Chem.GetFormalCharge(mol), Descriptors.NumRadicalElectrons(mol) + 1
mol = Chem.AddHs(mol)
print(f"{formula} | canonical {canon} | InChIKey {ikey} | charge {charge} | multiplicity {mult}")

ff = "MMFF94"
if not AllChem.MMFFHasAllMoleculeParams(mol):
    ff = "UFF"
    print("WARNING: MMFF94 has no parameters for this molecule; falling back to UFF. Record this.")
    assert AllChem.UFFHasAllMoleculeParams(mol), "UFF cannot parameterize this molecule either"

params = AllChem.ETKDGv3()
params.randomSeed = SEED
params.pruneRmsThresh = -1.0          # keep every embedding; deduplicate after minimization
cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=NCONFS, params=params))
assert cids, "embedding failed; try params.useRandomCoords = True"
print(f"embedded: {len(cids)}")

# torsion audit on the raw embeddings (before minimization)
audit = []
for atoms, label in torsion_list(mol):
    vals = [abs(rdMolTransforms.GetDihedralDeg(mol.GetConformer(c), *atoms)) for c in cids]
    lo, hi = sum(v < 90 for v in vals), sum(v >= 90 for v in vals)
    audit.append((label, atoms, lo, hi))

if ff == "MMFF94":
    results = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=MAX_ITERS)
else:
    results = AllChem.UFFOptimizeMoleculeConfs(mol, maxIters=MAX_ITERS)
flags = {cid: flag for cid, (flag, e) in zip(cids, results)}
energies = {cid: e for cid, (flag, e) in zip(cids, results) if flag == 0}
n_conv, n_unconv, n_failed = len(energies), sum(f == 1 for f in flags.values()), sum(f == -1 for f in flags.values())
print(f"{ff}: converged {n_conv}, not converged {n_unconv}, could not set up {n_failed}")
assert energies, "no conformer converged; raise MAX_ITERS"

kept, cluster_of, merged_mirror = unique_conformers(mol, energies)
e_min = energies[kept[0]]
survivors = [c for c in kept if (energies[c] - e_min) * KCAL_TO_KJ <= WINDOW_KJ]
print(f"distinct minima: {len(kept)} (mirror images {'merged' if merged_mirror else 'kept: chiral molecule'})")
print(f"survivors within {WINDOW_KJ} kJ/mol: {len(survivors)}")

RDKit 2026.03.6 | molecule: p-hydrobenzoic acid | seed: 230970 | requested: 100
C7H6O3 | canonical O=C(O)c1ccc(O)cc1 | InChIKey FJKROLUGYXJWQN-UHFFFAOYSA-N | charge 0 | multiplicity 1
embedded: 100
MMFF94: converged 100, not converged 0, could not set up 0
distinct minima: 2 (mirror images merged)
survivors within 12.6 kJ/mol: 2


In [4]:
with open(f"{NAME}_ensemble.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["conf_id", "ff_flag", "converged", "E_kcalmol", "dE_kJmol", "cluster_rep", "is_distinct"])
    for cid in sorted(cids, key=lambda c: energies.get(c, math.inf)):
        e = energies.get(cid)
        w.writerow([cid, flags[cid], flags[cid] == 0,
                    f"{e:.4f}" if e is not None else "",
                    f"{(e - e_min) * KCAL_TO_KJ:.3f}" if e is not None else "",
                    cluster_of.get(cid, ""), cid in kept])

pop = np.array([math.exp(-(energies[c] - e_min) / (R_KCAL * 298.15)) for c in kept]); pop /= pop.sum()
with open(f"{NAME}_survivors.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["rank", "conf_id", "E_kcalmol", "dE_kJmol", "boltzmann_298K", "in_window"]
               + [label for label, a, lo, hi in audit])
    for rank, cid in enumerate(kept, 1):
        row = [rank, cid, f"{energies[cid]:.4f}", f"{(energies[cid] - e_min) * KCAL_TO_KJ:.3f}",
               f"{pop[rank - 1]:.4f}", cid in survivors]
        row += [int(round(rdMolTransforms.GetDihedralDeg(mol.GetConformer(cid), *a))) for label, a, lo, hi in audit]
        w.writerow(row)

for rank, cid in enumerate(survivors, 1):
    if rank > MAX_INP:
        print(f"WARNING: {len(survivors)} survivors; input files written for the first {MAX_INP} only. "
              f"Narrow the window or justify carrying more.")
        break
    title = f"{NAME} conf{rank:02d} {ff} {energies[cid]:.3f} kcal/mol seed {SEED} rdkit {rdkit.__version__}"
    open(f"{NAME}_conf{rank:02d}.inp", "w").write(orca_input(mol, cid, title))
    Chem.MolToXYZFile(mol, f"{NAME}_conf{rank:02d}.xyz", confId=cid)

with open(f"{NAME}_search_record.txt", "w") as f:
    f.write(f"rdkit {rdkit.__version__}\npython {sys.version.split()[0]}\n")
    f.write(f"smiles_used {SMILES}\ncanonical_smiles {canon}\ninchikey {ikey}\nformula {formula}\n")
    f.write(f"charge {charge}\nmultiplicity {mult}\nforce_field {ff}\nseed {SEED}\n")
    f.write(f"n_confs_requested {NCONFS}\nn_confs_embedded {len(cids)}\nn_ff_converged {n_conv}\n")
    f.write(f"n_ff_not_converged {n_unconv}\nn_ff_failed {n_failed}\n")
    f.write(f"rms_thresh_angstrom {RMS_THRESH}\nmirror_images_merged {merged_mirror}\n")
    f.write(f"n_distinct_minima {len(kept)}\nwindow_kJmol {WINDOW_KJ}\nn_survivors_window {len(survivors)}\n")
    f.write(f"ff_min_kcalmol {e_min:.4f}\n")
    f.write("torsion_audit (label | dihedral atoms | count |phi|<90 | count |phi|>=90 ; a 0 means never sampled)\n")
    for label, a, lo, hi in audit:
        f.write(f"  {label} {a} {lo} {hi}\n")

print("\nsurvivors (rank, conf_id, E kcal/mol, dE kJ/mol, p298):")
for rank, cid in enumerate(survivors, 1):
    print(f"  {rank:2d} {cid:4d} {energies[cid]:10.4f} {(energies[cid] - e_min) * KCAL_TO_KJ:7.3f} {pop[rank - 1]:6.3f}")
print("\ntorsion audit (raw embeddings, |phi| < 90 / >= 90):")
for label, a, lo, hi in audit:
    flag = "   <- never sampled" if 0 in (lo, hi) else ""
    print(f"  {label:32s} {lo:3d} / {hi:3d}{flag}")
print(f"\nwrote {NAME}_ensemble.csv, {NAME}_survivors.csv, {NAME}_search_record.txt, "
      f"{min(len(survivors), MAX_INP)} x (.inp, .xyz)")
print(f"validate every .inp with: python3 check_orca_input.py {NAME}_conf01.inp")


survivors (rank, conf_id, E kcal/mol, dE kJ/mol, p298):
   1   19     0.0461   0.000  0.545
   2   74     0.1524   0.445  0.455

torsion audit (raw embeddings, |phi| < 90 / >= 90):
  rotatable bond 1-3                51 /  49
  hydroxyl H-O0-1-2                100 /   0   <- never sampled
  hydroxyl H-O7-6-5                 52 /  48

wrote p-hydrobenzoic acid_ensemble.csv, p-hydrobenzoic acid_survivors.csv, p-hydrobenzoic acid_search_record.txt, 2 x (.inp, .xyz)
validate every .inp with: python3 check_orca_input.py p-hydrobenzoic acid_conf01.inp
